# Import packages and data loading

In [1]:
import time
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src import (
    load_instance,
    build_model,
    solve_model,
    solve_and_summarize_all,
    extract_assignment,
    print_summary,
    get_model_stats,
    display_solution,
)

import json
import pandas as pd

there are two type of datasets, the strictly-ranked datasets are ranking the students strictly, treating the problem as no ties ever occurs.

In [2]:
data_small = json.load(open(r'../data/generated/instance_small.json', 'r', encoding='utf-8'))
print("Loaded small instance:\n   n={}, m={}".format(data_small['n'], data_small['m']))

data_medium = json.load(open(r'../data/generated/instance_medium.json', 'r', encoding='utf-8'))
print("Loaded medium instance:\n   n={}, m={}".format(data_medium['n'], data_medium['m']))

data_strict_small = json.load(open(r'../data/generated/instance_strict_small.json', 'r', encoding='utf-8'))
print("Loaded strictly-ranked small instance:\n   n={}, m={}".format(data_strict_small['n'], data_strict_small['m']))

data_strict_medium = json.load(open(r'../data/generated/instance_strict_medium.json', 'r', encoding='utf-8'))
print("Loaded strictly-ranked medium instance:\n   n={}, m={}".format(data_strict_medium['n'], data_strict_medium['m']))

Loaded small instance:
   n=10, m=5
Loaded medium instance:
   n=50, m=20
Loaded strictly-ranked small instance:
   n=10, m=5
Loaded strictly-ranked medium instance:
   n=50, m=20


# Solving Models

## Using Simple Datasets
in this section we use simple datasets, where ties happen. since the formulations for these models isn't designed for such datasets, the formulations below can't find a feasible solution in case of ties:
- `SO-BB`
- `SO-NW-CUT`
- `SO-NW-BIN-CUT`

In [3]:
results_small = solve_and_summarize_all(data_small, solver_name='cplex', tee=False)


print("\nSUMMARY TABLE")
print("="*100)

df_summary = pd.DataFrame([
    {
        'Formulation': r['model'],
        'Status': r['status'].upper(),
        'Model Obj': f"{r['model_objective']:.2f}" if r['model_objective'] is not None else "N/A",
        'Rank Obj': f"{r['rank_objective']}" if r['rank_objective'] is not None else "N/A",
        'Matched': r['students_assigned'],
        'Avg Rank': f"{r['rank_objective']/max(1, r['students_assigned']):.2f}" if r['rank_objective'] is not None and r['students_assigned'] > 0 else "N/A"
    }
    for r in results_small
])

print(df_summary.to_string(index=False))

ERROR: evaluating object as numeric value: x[0,4]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[0,4]
ERROR: evaluating object as numeric value: x[1,3]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[1,3]
ERROR: evaluating object as numeric value: x[2,0]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[2,0]
ERROR: evaluating object as numeric value: x[3,2]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[3,2]
ERROR: evaluating object as numeric value: x[4,3]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[4,3]
ERROR: evaluating object as numeric value: x[5,1]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object x[5,1]
ERROR: evaluating object as numeric value: x[6

seeing the results in the summary table, we can understand the weakness of these methods.

## Using no-tie datasets
in the strictly-ranked datasets, we added an epsilon value to each student score to avoid having ties. using these datasets, we can see how all the five formulations find a matching with the same average ranking of assigned students.

In [4]:
results_small = solve_and_summarize_all(data_strict_small, solver_name='cplex', tee=False)

print("\nSUMMARY TABLE")
print("="*100)

df_summary = pd.DataFrame([
    {
        'Formulation': r['model'],
        'Status': r['status'].upper(),
        'Model Obj': f"{r['model_objective']:.2f}" if r['model_objective'] is not None else "N/A",
        'Rank Obj': f"{r['rank_objective']}" if r['rank_objective'] is not None else "N/A",
        'Matched': r['students_assigned'],
        'Avg Rank': f"{r['rank_objective']/max(1, r['students_assigned']):.2f}" if r['rank_objective'] is not None and r['students_assigned'] > 0 else "N/A"
    }
    for r in results_small
])

print(df_summary.to_string(index=False))


SUMMARY TABLE
  Formulation  Status Model Obj Rank Obj  Matched Avg Rank
        SO-BB OPTIMAL      7.00        7        8     0.88
    SO-NW-CUT OPTIMAL      7.00        7        8     0.88
      MIN-CUT OPTIMAL    541.00        7        8     0.88
     MSMR-CUT OPTIMAL     33.00        7        8     0.88
SO-NW-BIN-CUT OPTIMAL      7.00        7        8     0.88
  MIN-BIN-CUT OPTIMAL     19.00        7        8     0.88
 MSMR-BIN-CUT OPTIMAL     33.00        7        8     0.88
      MSMR-EF OPTIMAL     33.00        7        8     0.88


to make sure that all models have the same answer, we can run the cell below to see the details and assignments for each model.

In [5]:
# Analyze each successful solution
formulations = ["SO-BB", "SO-NW-CUT", "MIN-CUT", "MSMR-CUT", "SO-NW-BIN-CUT", "MIN-BIN-CUT", "MSMR-BIN-CUT", "MSMR-EF"]
solution_details = {}

for formulation in formulations:
    info = solve_model(data_strict_small, formulation=formulation, solver_name='cplex', tee=False)
    model = info['model']
    status = info['status']
    
    if 'optimal' in status or 'feasible' in status:
        metrics = display_solution(formulation, model, data_strict_small)
        solution_details[formulation] = metrics
    else:
        print(f"\n{formulation}: INFEASIBLE or DID NOT SOLVE")


Formulation: SO-BB
Objective value: 7.00
Students matched: 8/10
Unmatched students: 2
Average preference rank: 0.88
Max preference rank: 3
Total rank objective: 7
College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Assignments: {0: 1, 1: 0, 2: None, 3: None, 4: 2, 5: 1, 6: 4, 7: 3, 8: 3, 9: 0}

Formulation: SO-NW-CUT
Objective value: 7.00
Students matched: 8/10
Unmatched students: 2
Average preference rank: 0.88
Max preference rank: 3
Total rank objective: 7
College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Assignments: {0: 1, 1: 0, 2: None, 3: None, 4: 2, 5: 1, 6: 4, 7: 3, 8: 3, 9: 0}

Formulation: MIN-CUT
Objective value: 541.00
Students matched: 8/10
Unmatched students: 2
Average preference rank: 0.88
Max preference rank: 3
Total rank objective: 7
College loads: {0: 2, 1: 2, 2: 1, 3: 2, 4: 1}
Assignments: {0: 1, 1: 0, 2: None, 3: None, 4: 2, 5: 1, 6: 4, 7: 3, 8: 3, 9: 0}

Formulation: MSMR-CUT
Objective value: 33.00
Students matched: 8/10
Unmatched students: 2
Average preference rank: 0.88
M

In [6]:
rows = []

for formulation in formulations:
    print(f"Solving {formulation} on strict medium...")
    start = time.perf_counter()
    info = solve_model(data_strict_medium, formulation=formulation, solver_name='cplex', tee=False)
    elapsed = time.perf_counter() - start
    model = info['model']
    stats = get_model_stats(model)
    rows.append({
        'Formulation': formulation,
        '#variables': stats['num_vars'],
        '#constraints': stats['num_constraints'],
        'size(Kb)': f"{stats['size_kb']:.2f}",
        'run time(s)': f"{elapsed:.2f}"
    })

# create data frame and print
df_table = pd.DataFrame(rows)
print("\nTable 2: The performance of (mixed) integer programming formulations for the classical Gale-Shapley model.")
print("="*100)
print(df_table.to_string(index=False))

Solving SO-BB on strict medium...
not match specified file format (lp)
Solving SO-NW-CUT on strict medium...
not match specified file format (lp)
Solving MIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-CUT on strict medium...
not match specified file format (lp)
Solving SO-NW-BIN-CUT on strict medium...
not match specified file format (lp)
Solving MIN-BIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-BIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-EF on strict medium...
not match specified file format (lp)

Table 2: The performance of (mixed) integer programming formulations for the classical Gale-Shapley model.
  Formulation  #variables  #constraints size(Kb) run time(s)
        SO-BB        1000          1070   337.51        0.95
    SO-NW-CUT        1040          2110   246.70        1.00
      MIN-CUT        1020          2070   225.53        0.73
     MSMR-CUT        1020          2070   236.94

In [7]:
data_strict_large = json.load(open(r'../data/generated/instance_strict_large.json', 'r', encoding='utf-8'))
print("Loaded strictly-ranked large instance:\n   n={}, m={}".format(data_strict_large['n'], data_strict_large['m']))

Loaded strictly-ranked large instance:
   n=1000, m=20


In [9]:
rows = []

for formulation in formulations:
    print(f"Solving {formulation} on strict large...")
    start = time.perf_counter()
    info = solve_model(data_strict_large, formulation=formulation, solver_name='cplex', tee=False)
    elapsed = time.perf_counter() - start
    model = info['model']
    stats = get_model_stats(model)
    rows.append({
        'Formulation': formulation,
        '#variables': stats['num_vars'],
        '#constraints': stats['num_constraints'],
        'size(Kb)': f"{stats['size_kb']:.2f}",
        'run time(s)': f"{elapsed:.2f}"
    })

# create data frame and print
df_table = pd.DataFrame(rows)
print("\nTable 2: The performance of (mixed) integer programming formulations for the classical Gale-Shapley model.")
print("="*100)
print(df_table.to_string(index=False))

Solving SO-BB on strict large...
not match specified file format (lp)
Solving SO-NW-CUT on strict large...
not match specified file format (lp)
Solving MIN-CUT on strict large...
not match specified file format (lp)
Solving MSMR-CUT on strict large...
not match specified file format (lp)
Solving SO-NW-BIN-CUT on strict large...
not match specified file format (lp)
Solving MIN-BIN-CUT on strict large...
not match specified file format (lp)
Solving MSMR-BIN-CUT on strict large...
not match specified file format (lp)
Solving MSMR-EF on strict large...
not match specified file format (lp)

Table 2: The performance of (mixed) integer programming formulations for the classical Gale-Shapley model.
  Formulation  #variables  #constraints   size(Kb) run time(s)
        SO-BB       20000         21020   95854.01      598.69
    SO-NW-CUT       20040         41060    5751.49      115.29
      MIN-CUT       20020         41020    5250.92      137.43
     MSMR-CUT       20020         41020    5570.